# Isharah-500 BiGRU + CTC training

This notebook documents the actual SI training pipeline and configurations. It is intentionally non-executing by default: the cells construct configuration objects or display commands, but do not call training. No model is retrained from this notebook.

## Architecture

The production model is `BiGRUCTC` from `isharah500/src/isharah_ctc.py`. It receives `(B, T, 2, 63)` keypoints, applies the hand-detection mask, flattens the hand features, projects them, encodes them with a bidirectional GRU, and emits frame-level CTC logits. The CTC blank token and vocabulary are restored from the checkpoint.

In [ ]:
from pathlib import Path
import os
import sys
def resolve_project_root():
    configured = os.environ.get("ISHARAH_PROJECT_ROOT")
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "isharah500" / "src").is_dir():
            return candidate
    raise RuntimeError("Set ISHARAH_PROJECT_ROOT to the repository worktree root.")

PROJECT_ROOT = resolve_project_root()
SOURCE_ROOT = PROJECT_ROOT / "isharah500" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from isharah_ctc import BiGRUCTCConfig
from isharah_training import TrainingConfig
model_config = BiGRUCTCConfig(num_hands=2, features_per_hand=63, projection_size=128, hidden_size=128, num_layers=2, dropout=0.2)
print(model_config.to_dict())

## Recorded SI configurations

The histories and checkpoint metadata record the following baseline settings: AdamW, batch size 8, learning rate `1e-3`, weight decay `1e-4`, gradient clipping `5.0`, seed `42`, zero data-loader workers, and hand-centric normalization for the hand-centric runs. The pilot used the original preprocessing; the five-epoch and resumed runs used hand-centric normalization.

The selected model is the 15-epoch hand-centric resume. Resume checkpoints preserve model state, optimizer state, epoch, validation state, random states, vocabulary, preprocessing, and training configuration.

In [ ]:
# Configuration-only example. This cell does not start training.
config = TrainingConfig(project_root=str(PROJECT_ROOT), data_root=os.environ.get("ISHARAH_DATA_ROOT"), output_dir="isharah500/checkpoints/example_run", batch_size=8, epochs=15, learning_rate=1e-3, weight_decay=1e-4, gradient_clip=5.0, seed=42, coordinate_normalization="hand_centric", device="cuda")
print(config.to_dict())
print("No training call was made.")

## Reproducible command templates

Run these from the project root in the verified Windows virtual environment. They are displayed for handover; this notebook does not execute them. Use a new output directory for any new run and never use the SI test labels for model selection.

In [ ]:
WINDOWS_TRAINING_COMMAND = r"""& "E:\DEPI Project\SIGNOVA_ESL\.venv\Scripts\python.exe" .\isharah500\src\isharah_training.py --project-root "E:\DEPI Project\SIGNOVA_ESL-isharah500" --data-root "E:\DEPI Project\SIGNOVA_ESL" --output-dir ".\isharah500\checkpoints\new_si_run" --device cuda --batch-size 8 --epochs 15 --learning-rate 1e-3 --weight-decay 1e-4 --gradient-clip 5.0 --seed 42 --coordinate-normalization hand_centric --train"""
WINDOWS_RESUME_COMMAND = r"""& "E:\DEPI Project\SIGNOVA_ESL\.venv\Scripts\python.exe" .\isharah500\src\isharah_training.py --project-root "E:\DEPI Project\SIGNOVA_ESL-isharah500" --data-root "E:\DEPI Project\SIGNOVA_ESL" --output-dir ".\isharah500\checkpoints\new_si_resume" --device cuda --epochs 15 --resume-from ".\isharah500\checkpoints\si_hand_centric_resume_10ep\isharah500_si_epoch_010.pt" --coordinate-normalization hand_centric --train"""
print(WINDOWS_TRAINING_COMMAND)
print(WINDOWS_RESUME_COMMAND)
print("Commands are displayed only; no subprocess is started.")

## Checkpoint contents

`save_checkpoint()` in `isharah_training.py` stores model weights, optimizer state, model configuration, vocabulary, preprocessing configuration, training configuration, metrics, early-stopping state, data validation, and random states. `load_checkpoint()` reconstructs the model and restores the serialized vocabulary and preprocessing contract for inference.

In [ ]:
# Optional read-only checkpoint metadata inspection. It does not train or modify the file.
import torch
checkpoint = PROJECT_ROOT / "isharah500" / "checkpoints" / "si_hand_centric_resume_15ep" / "isharah500_si_best.pt"
if checkpoint.is_file():
    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
    print({"epoch": payload.get("epoch"), "model_config": payload.get("model_config"), "preprocessing_config": payload.get("preprocessing_config"), "training_config": payload.get("training_config"), "validation_metrics": payload.get("validation_metrics")})
else:
    print("Selected checkpoint is not available locally; use the artifact handoff.")